# Bai existing candidate re-evaluation

Re-evaluates the first real trained Bai adapter under the corrected shared SFT/eval prompt contract. No retraining, no release creation, unchanged promotion gate. The frozen holdout remains unchanged. The run also audits production action payloads/truth-boundary discipline and records the current deterministic Character baseline. A metric PASS cannot route toward release when the candidate violates the action/truth contract.


In [ ]:
import os, sys, subprocess, json, shutil, zipfile
from pathlib import Path
print('Python', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51,<5', 'peft>=0.15,<1', 'datasets>=3,<5', 'accelerate', 'bitsandbytes', 'sentencepiece', 'huggingface-hub', 'kagglehub'], check=True)
import kagglehub
print('kagglehub ready')


In [ ]:
ROOT=Path('/kaggle/working/tamdeshevle')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1','https://github.com/eneonstudio-dev/tamdeshevle.git',str(ROOT)],check=True)
REPO_HEAD=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('repo', REPO_HEAD)

EXPECTED_BLOBS={
    'teacher-lab/corpus-builder.mjs':'4852330384c374dc733607eb8c72904ba2ae06d9',
    'teacher-lab/training/deterministic-seed.mjs':'3c581006e85071bacfcdc0ed25e63041ded1eb35',
    'teacher-lab/training/student-prompt-contract.json':'34010cf201709d3fb638f96954db587b5cbb2ffd',
    'teacher-lab/training/student-v0.1.json':'daf66037d81419d6605dd56bd95d0f65cb5d5c37',
    'teacher-lab/training/evaluate_student.py':'0770b36de1a851da391590a704dad60d2f8b18a7',
    'teacher-lab/benchmark.mjs':'721371a572868610d344becb0840bbd7f9900b34',
    'teacher-lab/training/benchmark-cli.mjs':'5b4c00a68e83b4c94ad55ca5ec8c673ea6963386',
    'teacher-lab/training/promotion-gate.mjs':'02e8fd1c7d46a2c9de80a33feb612902785c08ec',
    'teacher-lab/training/promotion-cli.mjs':'c0817973cc696745aa61dd46e5ae1e3cd2a3889c',
    'teacher-lab/training/reevaluate_candidate.py':'30f295825a39596ea568d225164ccb5402609d62',
}
for path,expected in EXPECTED_BLOBS.items():
    actual=subprocess.check_output(['git','-C',str(ROOT),'rev-parse',f'HEAD:{path}'],text=True).strip()
    if actual!=expected:
        raise RuntimeError(f'corrected re-eval contract drift: {path} expected={expected} actual={actual}')
print('corrected re-eval core contract pinned', len(EXPECTED_BLOBS))


In [ ]:
SOURCE='eneonstii/notebook07f42bd563/versions/1'
downloaded=Path(kagglehub.notebook_output_download(SOURCE))
print('downloaded', downloaded)
zips=list(downloaded.rglob('bai-candidate-experiment.zip'))
if not zips: raise FileNotFoundError('bai-candidate-experiment.zip not found in Version 1 output')
candidate_root=Path('/kaggle/working/first-bai-candidate')
if candidate_root.exists(): shutil.rmtree(candidate_root)
candidate_root.mkdir(parents=True)
with zipfile.ZipFile(zips[0]) as z: z.extractall(candidate_root)
adapter=candidate_root/'adapter'
if not adapter.is_dir(): raise FileNotFoundError(f'adapter missing after extract: {adapter}')
print('adapter', adapter)


In [ ]:
seed=Path('/kaggle/working/bai-reeval-seed')
if seed.exists(): shutil.rmtree(seed)
subprocess.run(['node',str(ROOT/'teacher-lab/training/deterministic-seed.mjs'),str(seed)],cwd=ROOT,check=True)
eval_gold=seed/'eval-gold.jsonl'
heldout_count=sum(1 for line in eval_gold.open(encoding='utf-8') if line.strip())
if heldout_count!=60: raise RuntimeError(f'frozen heldout count drift: {heldout_count}')
print('heldout rows', heldout_count)


In [ ]:
out=Path('/kaggle/working/bai-candidate-reeval')
if out.exists(): shutil.rmtree(out)
cmd=[sys.executable,str(ROOT/'teacher-lab/training/reevaluate_candidate.py'),'--eval-gold',str(eval_gold),'--candidate-adapter',str(adapter),'--out',str(out)]
result=subprocess.run(cmd,cwd=ROOT)
print('reeval exit', result.returncode)
manifest=json.loads((out/'reeval-manifest.json').read_text(encoding='utf-8'))
if manifest.get('status') not in {'REEVAL_PASS','REEVAL_REJECTED'}:
    raise RuntimeError(f"re-evaluation did not reach final status: {manifest.get('status')}")
print(json.dumps(manifest,ensure_ascii=False,indent=2))


In [ ]:
baseline_pred=out/'baseline-predictions.jsonl'
candidate_pred=out/'candidate-predictions.jsonl'
if not baseline_pred.is_file() or not candidate_pred.is_file():
    raise FileNotFoundError('corrected baseline/candidate predictions missing')

audit_dir=out/'contract-audit'
audit_dir.mkdir(parents=True,exist_ok=True)
for label,pred in [('baseline',baseline_pred),('candidate',candidate_pred)]:
    audit_out=audit_dir/f'{label}.json'
    audit_cmd=[sys.executable,str(ROOT/'teacher-lab/training/candidate_contract_audit.py'),'--eval-gold',str(eval_gold),'--predictions',str(pred),'--out',str(audit_out)]
    audit_result=subprocess.run(audit_cmd,cwd=ROOT)
    print(label,'contract audit exit',audit_result.returncode)
    print(audit_out.read_text(encoding='utf-8'))

deterministic_character=out/'deterministic-character-baseline.json'
character_result=subprocess.run(['node',str(ROOT/'scripts/test-bai-current-runtime-character-probe.mjs')],cwd=ROOT,capture_output=True,text=True)
if character_result.returncode!=0:
    print(character_result.stdout)
    print(character_result.stderr,file=sys.stderr)
    raise RuntimeError('deterministic release-safe Character baseline failed')
deterministic_character.write_text(character_result.stdout,encoding='utf-8')
character_report=json.loads(character_result.stdout)
if character_report.get('ok') is not True:
    raise RuntimeError('deterministic Character baseline report is not PASS')
print('deterministic Character baseline PASS')


In [ ]:
decision_file=out/'reeval-decision-summary.json'
decision_cmd=[
    sys.executable,str(ROOT/'teacher-lab/training/reeval_decision.py'),
    '--reeval-manifest',str(out/'reeval-manifest.json'),
    '--baseline-audit',str(out/'contract-audit/baseline.json'),
    '--candidate-audit',str(out/'contract-audit/candidate.json'),
    '--deterministic-character',str(deterministic_character),
    '--out',str(decision_file),
]
subprocess.run(decision_cmd,cwd=ROOT,check=True)
decision=json.loads(decision_file.read_text(encoding='utf-8'))
print(json.dumps(decision,ensure_ascii=False,indent=2))


In [ ]:
fail=out/'failure-analysis'
subprocess.run([sys.executable,str(ROOT/'teacher-lab/training/analyze_candidate_failures.py'),'--eval-gold',str(eval_gold),'--predictions',str(candidate_pred),'--out-dir',str(fail)],cwd=ROOT,check=True)
print((fail/'failure-summary.json').read_text(encoding='utf-8'))

comparison=out/'comparison'
runs=[]
historical_root=None
for historical_candidate in sorted(downloaded.rglob('candidate-predictions.jsonl')):
    root=historical_candidate.parent
    required=[root/'baseline-predictions.jsonl',root/'metrics'/'baseline.json',root/'metrics'/'candidate.json']
    if all(p.is_file() for p in required):
        historical_root=root; break
if historical_root is not None:
    print('historical comparison root', historical_root)
    runs += ['--run','historical-baseline',str(historical_root/'baseline-predictions.jsonl'),str(historical_root/'metrics'/'baseline.json')]
    runs += ['--run','historical-candidate',str(historical_root/'candidate-predictions.jsonl'),str(historical_root/'metrics'/'candidate.json')]
else:
    print('Historical prediction files not present in Version 1 output; comparing corrected runs only')
runs += ['--run','corrected-baseline',str(out/'baseline-predictions.jsonl'),str(out/'metrics'/'baseline.json')]
runs += ['--run','corrected-candidate',str(out/'candidate-predictions.jsonl'),str(out/'metrics'/'candidate.json')]
compare_cmd=[sys.executable,str(ROOT/'teacher-lab/training/compare_candidate_runs.py'),'--eval-gold',str(eval_gold),*runs,'--out-dir',str(comparison)]
subprocess.run(compare_cmd,cwd=ROOT,check=True)
print((comparison/'candidate-comparison.md').read_text(encoding='utf-8'))


In [ ]:
handoff_prefix=Path('/kaggle/working/bai_candidate_reeval')
pack_cmd=[sys.executable,str(ROOT/'teacher-lab/training/package_reeval_artifact.py'),'--reeval-dir',str(out),'--eval-gold',str(eval_gold),'--candidate-adapter',str(adapter),'--out-prefix',str(handoff_prefix),'--source-ref',SOURCE]
subprocess.run(pack_cmd,cwd=ROOT,check=True)
handoff_file=Path(str(handoff_prefix)+'-handoff.json')
handoff=json.loads(handoff_file.read_text(encoding='utf-8'))
print(json.dumps(handoff,ensure_ascii=False,indent=2))
print('Download these handoff artifacts:')
print(handoff['evidence_zip'])
print(handoff['adapter_zip'])
print(handoff_file)
